# 模型性能测试与可视化

本笔记用于复测最新代码中 4 个模型/流程的性能：
- KBTL Binary
- KBTL Multiclass
- MJDA (Gnat repair)
- BDA (Gnat-Piper demo 的目标训练集性能)


In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from scipy.io import loadmat
from sklearn.preprocessing import StandardScaler

from kernels.kernelRBF import kernelRBF
from models.kbtl.kbtl_train_binary import kbtl_train_binary
from models.kbtl.kbtl_test_binary import kbtl_test_binary
from models.kbtl.kbtl_train import kbtl_train
from models.kbtl.kbtl_test import kbtl_test
from models.mjda import mjda
from models.domainAdaptationTransform import domainAdaptationTransform
from classifiers.classifierKNN_cv import classifierKNN_cv
from classifiers.classifierKNN import classifierKNN
from models.bda import bda
from util.accuracy import accuracy
from util.f1score import f1score


In [ ]:
# 运行评测（与 analysis/model_performance_results.json 一致）
results = {}

root = Path('.')

# --- KBTL Binary ---
D=loadmat(root/'data'/'kbtl_demo_binary_data.mat')
X=[D['X'][0][i] for i in range(D['X'].shape[1])]
Xtest=[D['Xtest'][0][i] for i in range(D['Xtest'].shape[1])]
Y=[D['Y'][0][i].flatten() for i in range(D['Y'].shape[1])]
Ytest=[D['Ytest'][0][i].flatten() for i in range(D['Ytest'].shape[1])]
Ktrain=[]; khyp=[]
for t in range(len(X)):
    K,h=kernelRBF(np.nan,X[t],X[t]); Ktrain.append(K); khyp.append(h)
params={'lambda': {'kappa': 1e-3, 'theta': 1e-3},'gamma': {'kappa': 1e-3, 'theta': 1e-3},'eta': {'kappa': 1e-3, 'theta': 1e-3},'iter': 500,'margin': 0,'R': 2,'Hsigma2': 0.25**2}
np.random.seed(0)
params=kbtl_train_binary(Ktrain,Y,params)
pred_train=kbtl_test_binary(Ktrain,params,Y)
Ktest=[]
for t in range(len(X)):
    K,_=kernelRBF(khyp[t],X[t],Xtest[t]); Ktest.append(K)
pred_test=kbtl_test_binary(Ktest,params,Ytest)
results['kbtl_binary']={'mean_train_acc':float(np.mean(pred_train['acc'])),'mean_test_acc':float(np.mean(pred_test['acc'])),'test_acc':list(map(float,pred_test['acc']))}

# --- KBTL Multiclass ---
D=loadmat(root/'data'/'kbtl_demo_multiclass_data.mat')
X=D['X'][0]; Y=D['Y'][0]; Xtest=D['Xtest'][0]; Ytest=D['Ytest'][0]
params={'lambda': {'kappa': 1e-3, 'theta': 1e-3},'gamma': {'kappa': 1e-3, 'theta': 1e-3},'eta': {'kappa': 1e-3, 'theta': 1e-3},'iter': 500,'margin': 1,'R': 2,'Hsigma2': 6**2}
Ktrain=[];kh=[]
for t in range(len(X)):
    K,h=kernelRBF(np.nan,X[t],X[t]); Ktrain.append(K); kh.append(h)
np.random.seed(0)
hyp,_=kbtl_train(Ktrain,Y,params)
pred_test=kbtl_test([kernelRBF(kh[t],X[t],Xtest[t])[0] for t in range(len(X))],hyp,Ytest)
results['kbtl_multiclass']={'mean_test_acc':float(np.mean(pred_test['acc'])),'test_acc':list(map(float,pred_test['acc']))}

# --- MJDA ---
data=loadmat(root/'data'/'gnat_repair.mat')
Xs=data['Xs'];Ys=data['Ys'].flatten();Xt=data['Xt'];Yt=data['Yt'].flatten()
np.random.seed(0)
idx=np.random.permutation(len(Xs)); tr=idx[:500]; te=idx[500:]
Xs_tr,Xs_te=Xs[tr],Xs[te]; Ys_tr=Ys[tr]
Xt_tr,Xt_te=Xt[tr],Xt[te]; Yt_te=Yt[te]
Xs_tr=StandardScaler().fit_transform(Xs_tr); Xs_te=StandardScaler().fit(Xs[tr]).transform(Xs_te)
Xt_tr=StandardScaler().fit_transform(Xt_tr); Xt_te=StandardScaler().fit(Xt[tr]).transform(Xt_te)
Zs,Zt,Ytp,W,cls,_,_=mjda(Xs_tr,Ys_tr,Xt_tr,kernelRBF,np.nan,0.1,2,classifierKNN_cv,1,8,1000,Yt[tr])
Zs_te=domainAdaptationTransform(Xs_te,Xs_tr,Xt_tr,W,kernelRBF,np.nan)
Zt_te=domainAdaptationTransform(Xt_te,Xs_tr,Xt_tr,W,kernelRBF,np.nan)
Ytp_te,_=classifierKNN_cv(Zs_te,Ys_tr,Zt_te,cls)
results['mjda']={'test_acc':float(accuracy(Ytp_te,Yt_te)),'test_f1':float(f1score(Yt_te,Ytp_te)[0])}

# --- BDA Gnat-Piper ---
feat=loadmat(root/'data'/'gnat_piper_preprocessed_features.mat')
lbl=loadmat(root/'data'/'gnat_piper_preprocessed_labels.mat')
Xs_cells=feat['Xs_tr']; Ys_cells=lbl['Ys_tr']
Xs_tr=np.vstack([Xs_cells[i,0] for i in range(Xs_cells.shape[0])])
Ys_tr=np.hstack([Ys_cells[i,0].flatten() for i in range(Ys_cells.shape[0])])
Xt_tr=feat['Xt_tr']; Yt_tr=lbl['Yt_tr'].flatten()
Xs_tr=StandardScaler().fit_transform(Xs_tr); Xt_tr=StandardScaler().fit_transform(Xt_tr)
_,_,Ytp,_,_,_,mmd=bda(Xs_tr,Ys_tr,Xt_tr,kern=kernelRBF,hyp=np.nan,mu=1.0,k=10,lambda_=0.5,classifier=classifierKNN,iter=2,mode=0,Yt=Yt_tr)
results['bda_gnat_piper']={'train_target_acc':float(accuracy(Yt_tr,Ytp)),'train_target_f1':float(f1score(Yt_tr,Ytp)[0]),'mmd':float(mmd)}

results


## 本次运行结果（已保存）

- 结果 JSON：`analysis/model_performance_results.json`
- 图 1：`analysis/model_performance_bar.svg`
- 图 2：`analysis/kbtl_task_accuracy.svg`

### 指标摘要

- KBTL Binary 平均测试准确率：**97.24%**
- KBTL Multiclass 平均测试准确率：**89.90%**
- MJDA 测试准确率 / F1：**100.00% / 1.0000**
- BDA(Gnat-Piper) 目标训练准确率 / F1：**69.21% / 0.6112**


In [ ]:
from IPython.display import SVG, display

display(SVG(filename='analysis/model_performance_bar.svg'))
display(SVG(filename='analysis/kbtl_task_accuracy.svg'))
